# Lab 2 – Model selection: validation, cross-validation and pipelines

**Machine Learning** – Master in Data Science and Business Analytics – A.Y. 2026/27
Lab tutor: Francesco Casadei

### Where we are
In Lab 1 we trained decision trees on *Bank Marketing* and chose `max_depth` **by looking at the test set**. Today we fix this: the test set must be used **once**, at the very end, to estimate how the chosen model will perform on new clients. All the choices (hyperparameters, preprocessing) are made with the training data only.

> *Business question* (unchanged): **before calling**, can we predict which clients are likely to subscribe a term deposit?

### Goals of the lab
1. finish Lab 1: limiting the leaf size (ex. 6.3) and the `duration` leakage (sec. 7)
2. see how much a single holdout split can fool us
3. tune a hyperparameter with a **train / validation / test** split
4. estimate performance with **k-fold cross-validation** (`cross_val_score`)
5. put preprocessing and model in a scikit-learn **`Pipeline`** with a **`ColumnTransformer`**
6. tune several hyperparameters at once with **`GridSearchCV`**, then evaluate the winner on the test set

Indicative timing: setup 5' · finishing Lab 1 20' · one split is not enough 15' · train/validation/test 15' · cross-validation 20' · pipeline 15' · grid search 25' · wrap-up 5'

Sections marked *optional* are for those who finish early or for homework.

## 0. Setup
Put `bank-additional-full.csv` in the same folder as this notebook (on Google Colab: upload it with the *Files* panel on the left).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV, learning_curve)
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
DATA_PATH = 'bank-additional-full.csv'
sns.set_theme(style='whitegrid')

### 0.1 Data preparation (recap of Lab 1)
The cell below repeats, in compact form, the preparation of Lab 1: duplicates removed, `duration` dropped (leakage), `prev_contacted` added, target encoded as 0/1. Run it and check that you obtain the same shapes as last time.

We keep two versions of the predictors:
- `X_raw`: the categorical attributes are still **strings** – we will need it for the pipeline in section 5
- `X`: one-hot encoded with `pd.get_dummies`, as in Lab 1

In [ ]:
df = pd.read_csv(DATA_PATH, sep=';')
df = df.drop_duplicates()
df_leak = df.copy()                                    # still contains 'duration' (section 1.2)

df = df.drop(columns=['duration'])
df['prev_contacted'] = (df['pdays'] != 999).astype(int)

y = (df['y'] == 'yes').astype(int)
X_raw = df.drop(columns=['y'])

num_cols = X_raw.select_dtypes(include='number').columns.tolist()
cat_cols = [c for c in X_raw.columns if c not in num_cols]
X = pd.get_dummies(X_raw, columns=cat_cols, dtype=int)

print('X_raw:', X_raw.shape, '  X (one-hot):', X.shape, '  rate of yes:', y.mean().round(3))
print('categorical:', cat_cols)

### 0.2 The holdout split of Lab 1
Same 70/30 stratified split and same `random_state` as Lab 1, so the numbers are comparable. We also recompute the baseline.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

baseline_acc = DummyClassifier(strategy='most_frequent').fit(X_train, y_train).score(X_test, y_test)
print(X_train.shape, X_test.shape)
print(f'Baseline test accuracy: {baseline_acc:.4f}')

## 1. Warm-up: finishing Lab 1

### 1.1 Limiting the size of the leaves (Lab 1, ex. 6.3)
Overfitting can also be controlled with `min_samples_leaf`, the minimum number of training samples in a leaf. Train trees **without depth limit**, `criterion='entropy'`, for `min_samples_leaf` in the list below; for each one store training accuracy, test accuracy and number of leaves (`get_n_leaves()`). Then plot the two accuracies with a logarithmic *x* axis (`plt.xscale('log')`) together with the baseline.

In [ ]:
leaf_sizes = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
rows = []
for m in leaf_sizes:
    # TODO: tree with min_samples_leaf=m (no depth limit, criterion='entropy');
    #       append a dict with min_samples_leaf, n_leaves, train_acc, test_acc
    ...
leaf_res = pd.DataFrame(rows).set_index('min_samples_leaf')
display(leaf_res.round(4))

plt.figure(figsize=(9, 5))
# TODO: plot train_acc and test_acc vs min_samples_leaf, plus the baseline; log scale on x
...
plt.xlabel('min_samples_leaf'); plt.ylabel('accuracy'); plt.legend()
plt.title('Decision tree: accuracy vs min_samples_leaf');

**Questions**
1. Read the curves **from right to left**: why does `min_samples_leaf` act as a brake on the complexity of the tree, in the opposite direction of `max_depth`?
2. Which value gives the best test accuracy? How many leaves has that tree, compared with the fully grown one?
3. Once again we chose the "best" value by looking at the **test** set: keep this in mind for section 2.

### 1.2 What if we had kept `duration`? (Lab 1, sec. 7)
Repeat the preparation on `df_leak` (which still contains `duration`), using **the same rows** for training and test, and train a depth-5 tree with and without `duration`. Compare the test accuracies and the **share of real subscribers that each model identifies** (among the test clients with `y = 1`, the fraction predicted as `1`).

In [ ]:
df_leak['prev_contacted'] = (df_leak['pdays'] != 999).astype(int)
X_leak = pd.get_dummies(df_leak.drop(columns=['y']), columns=cat_cols, dtype=int)
X_leak_train, X_leak_test = X_leak.loc[X_train.index], X_leak.loc[X_test.index]   # same rows as before

for name, (Xa, Xb) in {'without duration': (X_train, X_test),
                       'with duration':    (X_leak_train, X_leak_test)}.items():
    # TODO: depth-5 tree trained on (Xa, y_train); predict on Xb;
    #       print test accuracy and the share of subscribers identified: pred[y_test == 1].mean()
    ...

**Questions**
1. The share of subscribers identified roughly doubles, while accuracy gains only about one point. Which of the two numbers tells the bank more? (→ *Lab 3: performance measures*)
2. The model with `duration` is clearly "better". Why would it be a serious mistake to deliver it to the bank?

## 2. How much can a single holdout split fool us?
The test accuracy depends on **which** clients ended up in the test set. Let us repeat the experiment of Lab 1 (accuracy vs `max_depth`) with 10 different random splits.

For each `seed` in `range(10)`: split `X, y` 70/30 (stratified, `random_state=seed`), train trees with `max_depth` from 1 to 15 and record (a) the depth with the highest test accuracy and (b) that accuracy.

In [ ]:
depths = range(1, 16)
rows = []
for seed in range(10):
    # TODO: stratified 70/30 split with random_state=seed
    Xa, Xb, ya, yb = ...
    # TODO: list with the test accuracy of a tree for each depth in depths
    accs = ...
    rows.append({'seed': seed, 'best_depth': depths[int(np.argmax(accs))], 'best_test_acc': max(accs)})
split_res = pd.DataFrame(rows).set_index('seed')
display(split_res.round(4))
print('best_depth values:', sorted(split_res['best_depth'].unique().tolist()))
print(f"best test accuracy: min {split_res['best_test_acc'].min():.4f}  max {split_res['best_test_acc'].max():.4f}")

**Questions**
1. Does the "best" depth stay the same? How large is the range of the best test accuracy, compared with the gain over the baseline (≈ 0.887)?
2. If two colleagues used two different seeds, would they agree on the model to deliver?
3. Each "best test accuracy" was obtained by choosing the depth on the test set itself. Is it an optimistic or a pessimistic estimate? Why?

## 3. Train / validation / test
First solution, as seen in class: split the **training** set again into a smaller training set and a **validation** set. Hyperparameters are chosen on the validation set; the test set is touched **once**, at the end.

```
all data ──► train (70%) ──► train_small (≈ 52%)  → fit the candidates
         │               └─► validation  (≈ 18%)  → choose max_depth
         └─► test (30%)                            → final estimate, used once
```

Split `X_train, y_train` into `X_tr, X_val` (75/25, stratified, `random_state=RANDOM_STATE`).

In [ ]:
# TODO: split the training set into train_small and validation (75/25, stratified)
X_tr, X_val, y_tr, y_val = ...
print(X_tr.shape, X_val.shape, X_test.shape)

For `max_depth` from 1 to 15 train on `(X_tr, y_tr)` and compute the accuracy on the validation set. Choose the best depth, **retrain** a tree with that depth on the whole training set `(X_train, y_train)` and only then compute its test accuracy.

In [ ]:
# TODO: validation accuracy for each depth (fit on X_tr, score on X_val)
val_acc = ...
best_depth_val = depths[int(np.argmax(val_acc))]

plt.figure(figsize=(9, 4))
plt.plot(depths, val_acc, 'o-', label='validation')
plt.axhline(baseline_acc, ls='--', c='k', label='baseline')
plt.axvline(best_depth_val, c='r', ls=':', label=f'best depth = {best_depth_val}')
plt.xlabel('max_depth'); plt.ylabel('accuracy'); plt.legend(); plt.title('Choosing max_depth on the validation set')
plt.show()

# TODO: retrain with best_depth_val on the WHOLE training set, then evaluate ONCE on the test set
final_tv = ...
...
print(f'validation accuracy at depth {best_depth_val}: {max(val_acc):.4f}')
print(f'test accuracy of the final model:      {final_tv.score(X_test, y_test):.4f}')

**Questions**
1. Why do we retrain on the whole training set before testing, instead of using the tree fitted on `X_tr`?
2. The choice now depends on a single validation set of about 7,000 clients. Does the problem of section 2 disappear?

## 4. k-fold cross-validation
With *k*-fold cross-validation the training set is split into *k* **folds**; each fold is used once for validation while the model is trained on the other *k*−1. We get *k* scores: their **mean** is the estimate, their **standard deviation** tells how much it depends on the particular data.

We use a [`StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html) with 5 folds and shuffling, defined once and reused everywhere, so that all the models are compared on **the same folds**.

### 4.1 `cross_val_score`
Use [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) to evaluate a depth-5 tree on the **training set** with the folds `cv5`. Print the 5 scores, their mean and standard deviation.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

tree5 = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=RANDOM_STATE)
# TODO: 5-fold cross-validated accuracy of tree5 on the training set
scores = ...
print('fold scores:', scores.round(4))
print(f'CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

**Question**: how many trees were trained by this single line? On how many rows was each of them trained?

### 4.2 Choosing `max_depth` with cross-validation
For `max_depth` from 1 to 15 compute the mean and the standard deviation of the CV accuracy, then plot the mean with an error band of ± 1 standard deviation (`plt.fill_between`).

In [ ]:
cv_mean, cv_std = [], []
for d in depths:
    # TODO: cross_val_score of a tree with max_depth=d on the training set (cv=cv5); store mean and std
    ...
cv_mean, cv_std = np.array(cv_mean), np.array(cv_std)
best_depth_cv = depths[int(np.argmax(cv_mean))]

plt.figure(figsize=(9, 5))
plt.plot(depths, cv_mean, 'o-', label='CV mean accuracy')
# TODO: band between cv_mean - cv_std and cv_mean + cv_std with plt.fill_between (alpha=0.25)
...
plt.axhline(baseline_acc, ls='--', c='k', label='baseline')
plt.axvline(best_depth_cv, c='r', ls=':', label=f'best depth = {best_depth_cv}')
plt.xlabel('max_depth'); plt.ylabel('accuracy'); plt.legend(); plt.title('5-fold CV accuracy vs depth');
print(f'best depth: {best_depth_cv}   CV accuracy {cv_mean.max():.4f} ± {cv_std[np.argmax(cv_mean)]:.4f}')

**Questions**
1. Compare the differences between the depths from 3 to 8 with the width of the band. Is the "best" depth really better than its neighbours?
2. When several values are practically equivalent, which one would you deliver to the bank, and why? (hint: think about interpretability and about the risk of overfitting)
3. Compared with the train/validation/test approach, what do we pay for this more reliable estimate?

## 5. Pipelines: preprocessing inside the model
So far we one-hot encoded **all** the data before splitting. For one-hot encoding this is harmless, but many preprocessing steps **learn statistics from the data** (the most frequent value to impute, the mean and standard deviation to scale, …). If they are computed on all the data, information from the validation/test rows leaks into training: **data leakage**, the subtle version.

The clean solution is a scikit-learn [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): a chain of preprocessing steps followed by a model, which behaves like a single estimator. When `cross_val_score` (or `GridSearchCV`) fits the pipeline on the training folds, **all the steps** are fitted on those folds only, and then applied to the validation fold.

A [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) applies different transformations to different columns:

```
X_raw ──► ColumnTransformer ─┬─ categorical columns ─► OneHotEncoder ─┐
                             └─ numeric columns ─────► passthrough  ──┴─► DecisionTreeClassifier
```

### 5.1 Build the pipeline
First, the same split as before, but on `X_raw` (same rows, selected through the index).

In [ ]:
X_raw_train, X_raw_test = X_raw.loc[X_train.index], X_raw.loc[X_test.index]
print(X_raw_train.shape, X_raw_test.shape)
X_raw_train.head(3)

Build:
- `preprocess`: a `ColumnTransformer` with a step `('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)` and `remainder='passthrough'` (the numeric columns go through unchanged)
- `pipe`: a `Pipeline` with two named steps, `'prep'` → `preprocess` and `'tree'` → a `DecisionTreeClassifier(criterion='entropy', random_state=RANDOM_STATE)`

*Note*: `handle_unknown='ignore'` avoids an error if a category never seen during training appears at prediction time (it happens with rare values and small folds).

In [ ]:
# TODO: ColumnTransformer: one-hot on cat_cols, numeric columns passed through
preprocess = ...

# TODO: Pipeline with steps 'prep' and 'tree'
pipe = ...
pipe

### 5.2 Use the pipeline like a model
Hyperparameters of a step are addressed as `<step name>__<parameter>` (two underscores). Set `max_depth=5` with `pipe.set_params(tree__max_depth=5)`, then compute its 5-fold CV accuracy on `X_raw_train` with the same folds `cv5`. You should obtain exactly the same scores of section 4.1: why?

In [ ]:
# TODO: set max_depth=5 in the 'tree' step and cross-validate the pipeline on X_raw_train
...
scores_pipe = ...
print('fold scores:', scores_pipe.round(4))
print(f'CV accuracy (pipeline): {scores_pipe.mean():.4f} ± {scores_pipe.std():.4f}')
print(f'CV accuracy (sec. 4.1): {scores.mean():.4f} ± {scores.std():.4f}')

**Questions**
1. `prev_contacted` was created **before** the split, outside the pipeline. Is this a leakage? (hint: does it use information from other rows?)
2. Give an example of a preprocessing step that, done before the split, *would* be a leakage.

## 6. Grid search: several hyperparameters at once
[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) automates what we did by hand in section 4.2: for **every combination** of the values in a grid it runs a cross-validation, picks the combination with the best mean score and (with `refit=True`, the default) retrains the pipeline with it on the **whole** training set.

### 6.1 Run the search
Define a grid on `tree__max_depth` (`[3, 4, 5, 6, 7, 8, 10, None]`) and `tree__min_samples_leaf` (`[1, 20, 50, 100, 200]`), and run `GridSearchCV` on the pipeline with `cv=cv5`, `scoring='accuracy'`, `n_jobs=-1` (use all the processor cores) and `return_train_score=True`.

**Before running**: how many trees will be trained? (it takes about half a minute)

In [ ]:
param_grid = {
    'tree__max_depth': [3, 4, 5, 6, 7, 8, 10, None],
    'tree__min_samples_leaf': [1, 20, 50, 100, 200],
}
# TODO: GridSearchCV on pipe with cv=cv5, scoring='accuracy', n_jobs=-1, return_train_score=True; fit on the training set
grid = ...
...

print('best parameters :', grid.best_params_)
print(f'best CV accuracy: {grid.best_score_:.4f}')

### 6.2 Look inside the results
`grid.cv_results_` contains, for every combination, the mean and standard deviation of the validation (and training) scores. Turn it into a dataframe, show the 10 best combinations, and draw a heatmap of `mean_test_score` with `max_depth` on the rows and `min_samples_leaf` on the columns.

In [ ]:
res = pd.DataFrame(grid.cv_results_)
cols = ['param_tree__max_depth', 'param_tree__min_samples_leaf',
        'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']
# TODO: show the 10 best combinations (sort by rank_test_score)
...

res['max_depth'] = [str(v) for v in res['param_tree__max_depth']]  # None → 'None', to use it as a label
# TODO: pivot_table with index='max_depth', columns='param_tree__min_samples_leaf', values='mean_test_score'
heat = ...
heat = heat.loc[[str(d) for d in param_grid['tree__max_depth']]]  # keep the order of the grid
plt.figure(figsize=(8, 5))
# TODO: sns.heatmap of heat with the values written in the cells (annot=True, fmt='.4f')
...
plt.xlabel('min_samples_leaf'); plt.ylabel('max_depth'); plt.title('5-fold CV accuracy');

**Questions**
1. Look at the row `max_depth = None`: how does the accuracy change with `min_samples_leaf`? Explain it with section 1.1.
2. Compare `mean_train_score` and `mean_test_score` for the best combinations and for `max_depth=None, min_samples_leaf=1`. Where do you see overfitting?
3. Several combinations have **exactly** the same scores. Why? (hint: with `min_samples_leaf=200`, can the tree actually grow deeper than 6–7 levels?) Which of the tied combinations would you prefer?
4. How large are the differences among the top 10, compared with `std_test_score`?

### 6.3 The final evaluation, once
`grid.best_estimator_` is the pipeline retrained on the whole training set with the best parameters (and `grid.predict` / `grid.score` use it). Compute its accuracy on the **test set**, and collect in a small table all the estimates obtained today.

In [ ]:
# TODO: test accuracy of the best pipeline found by the grid search
test_acc_final = ...
print(f'CV estimate (best combination): {grid.best_score_:.4f}')
print(f'test accuracy of the final model: {test_acc_final:.4f}')

summary = pd.DataFrame({
    'how the model was chosen': ['baseline (always "no")',
                                 'Lab 1 / sec. 1.1: best value on the test set',
                                 'sec. 3: train / validation / test',
                                 'sec. 6: grid search with 5-fold CV'],
    'test accuracy': [baseline_acc, leaf_res['test_acc'].max(),
                      final_tv.score(X_test, y_test), test_acc_final],
})
summary.round(4)

**Questions**
1. Is the test accuracy close to the CV estimate? Which of the two would you report to the bank as "expected accuracy on new clients"?
2. The row of section 1.1 may even look better than the grid search. Why is it nevertheless **not** a number we can trust?
3. Now that we have the test result, would it be correct to go back, change the grid and try again until the test accuracy improves?

## 7. Optional extensions

### 7.1 Preprocessing choices are hyperparameters too
In Lab 1 we kept `'unknown'` as a category of its own. An alternative is to treat it as a missing value and replace it with the most frequent value of the column, **learned on the training folds only**. Build a second pipeline whose categorical branch is itself a small pipeline: `SimpleImputer(missing_values='unknown', strategy='most_frequent')` followed by the `OneHotEncoder`. Compare the two options with cross-validation, using for both the best tree parameters found in section 6.

In [ ]:
# TODO: categorical branch = Pipeline(SimpleImputer(missing_values='unknown', strategy='most_frequent') → OneHotEncoder)
cat_branch = ...
# TODO: pipeline with a ColumnTransformer that uses cat_branch on cat_cols, then the tree
pipe_imp = ...

best_tree_params = {k: v for k, v in grid.best_params_.items()}
for name, p in {"'unknown' as a category": pipe, "'unknown' imputed": pipe_imp}.items():
    p.set_params(**best_tree_params)
    # TODO: cross-validated accuracy of p on the training set; print mean ± std
    ...

**Question**: is there any difference? When two options are equivalent, which one would you keep, and why?

*Hint*: you could also put both alternatives in a single `GridSearchCV`, passing a **list** of grids, e.g. `{'prep': [preprocess_a, preprocess_b], 'tree__max_depth': [...]}` – whole steps can be grid values.

### 7.2 Randomized search
When the grid becomes large, [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) tries only `n_iter` random combinations. Search `max_depth` in 2–20, `min_samples_leaf` in 1–300 and `criterion` in `['gini', 'entropy']` with `n_iter=25`. Does it find a result comparable to the grid search, with fewer fits?

In [ ]:
param_dist = {
    'tree__max_depth': list(range(2, 21)),
    'tree__min_samples_leaf': list(range(1, 301)),
    'tree__criterion': ['gini', 'entropy'],
}
# TODO: RandomizedSearchCV on pipe with n_iter=25, cv=cv5, random_state=RANDOM_STATE; fit on the training set
rand = ...
...
print('best parameters :', rand.best_params_)
print(f'best CV accuracy: {rand.best_score_:.4f}   (grid search: {grid.best_score_:.4f})')

### 7.3 Learning curve: would more data help?
A [`learning_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html) shows training and validation score as the size of the training set grows. Draw it for the best pipeline and for a fully grown tree (`max_depth=None, min_samples_leaf=1`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
models = {'best pipeline': grid.best_estimator_,
          'fully grown tree': Pipeline([('prep', preprocess),
                                        ('tree', DecisionTreeClassifier(criterion='entropy', random_state=RANDOM_STATE))])}
for ax, (name, model) in zip(axes, models.items()):
    # TODO: learning_curve(model, X_raw_train, y_train, cv=cv5, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 6))
    sizes, tr, va = ...
    # TODO: plot mean training and validation scores vs sizes, plus the baseline
    ...
    ax.set_title(name); ax.set_xlabel('number of training rows'); ax.legend()
axes[0].set_ylabel('accuracy');

**Question**: for which of the two models the gap between the curves is large (high *variance*)? For which one would collecting more data be more useful?

### 7.4 Teaser for Lab 3: optimising something other than accuracy
Re-run the grid search of section 6 with `scoring='recall'` (the share of real subscribers identified, as in section 1.2). Do the best parameters change? What happens to accuracy?

In [ ]:
# TODO: same grid search as section 6 with scoring='recall'; compare best parameters, test accuracy and test recall
...

## 8. Wrap-up
Discuss with your neighbour and write two lines for each question:
1. Why is the test accuracy of a model whose hyperparameters were chosen on the test set an optimistic estimate?
2. What does the standard deviation of the cross-validation scores tell you, and how should it influence the choice between two hyperparameter settings?
3. Why must preprocessing be **inside** the pipeline that is cross-validated?
4. All the accuracies we obtained today lie between 0.89 and 0.90, against a baseline of 0.887. Is accuracy the right yardstick for the bank's problem? → *Lab 3: performance measures*